# เฉลย Lab 06 — ประกวดโมเดลจัดหมวดมะเร็งเต้านม

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, export_text

X, y = load_breast_cancer(return_X_y=True)
names = list(load_breast_cancer().feature_names)
skf = StratifiedKFold(5, shuffle=True, random_state=42)

### ข้อ 1 — สนามประกวด (CV เดียวกัน 2 metric)

ตัวเลขอ้างอิงจากเครื่องของอาจารย์ (random_state เดียวกันควรได้เท่ากัน):
SVM 0.977/0.995 · LR 0.974/0.995 · KNN 0.963/0.985 · NB 0.930/0.986 · Tree อิสระ 0.910/0.900 · Dummy 0.627/0.500

In [2]:
zoo = [
    ("Dummy", DummyClassifier(strategy="most_frequent")),
    ("GaussianNB", make_pipeline(StandardScaler(), GaussianNB())),
    ("KNN(5)", make_pipeline(StandardScaler(), KNeighborsClassifier())),
    ("LogisticRegression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))),
    ("SVM(RBF)", make_pipeline(StandardScaler(), SVC(random_state=42))),
    ("Tree(free)", DecisionTreeClassifier(random_state=42)),
]
results = []
for name, m in zoo:
    acc = cross_val_score(m, X, y, cv=skf).mean()
    auc = cross_val_score(m, X, y, cv=skf, scoring="roc_auc").mean()
    results.append((auc, acc, name))
for auc, acc, name in sorted(results, reverse=True):
    print(f"{name:20s} acc={acc:.3f}  AUC={auc:.3f}")

LogisticRegression   acc=0.974  AUC=0.995
SVM(RBF)             acc=0.977  AUC=0.995
GaussianNB           acc=0.930  AUC=0.986
KNN(5)               acc=0.963  AUC=0.985
Tree(free)           acc=0.910  AUC=0.900
Dummy                acc=0.627  AUC=0.500


### ข้อ 2 — ต้นไม้เป็นภาษาไทย

path ตัวอย่างจากต้นไม้จริง (depth 3):
"ถ้า worst radius ≤ 16.8 และ worst concave points ≤ 0.14 และ area error ≤ 91.6 → class 1 (เนื้องอกดี)"
= ก้อนเนื้อเล็ก ผิวเรียบ ไม่มีการเปลี่ยนแปลงขนาดมาก → มักเป็นเนื้องอกดี

In [3]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
tree3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr, ytr)
print(export_text(tree3, feature_names=names, max_depth=2))
print("test accuracy:", round(tree3.score(Xte, yte), 3))

|--- worst radius <= 16.80
|   |--- worst concave points <= 0.14
|   |   |--- area error <= 91.56
|   |   |   |--- class: 1
|   |   |--- area error >  91.56
|   |   |   |--- class: 0
|   |--- worst concave points >  0.14
|   |   |--- worst texture <= 25.62
|   |   |   |--- class: 1
|   |   |--- worst texture >  25.62
|   |   |   |--- class: 0
|--- worst radius >  16.80
|   |--- texture error <= 0.47
|   |   |--- class: 1
|   |--- texture error >  0.47
|   |   |--- worst concavity <= 0.19
|   |   |   |--- class: 1
|   |   |--- worst concavity >  0.19
|   |   |   |--- class: 0

test accuracy: 0.944


### ข้อ 3 — ensemble effect

tree เดี่ยว 0.910 → RandomForest 0.956 → Voting(LR+RF+KNN) 0.972
ensemble "ยก" tree ขึ้นมาก แต่บนข้อมูลง่ายแบบนี้ single ที่ดี (SVM 0.977) ก็แซงได้ — ไม่มีใครชนะทุกที่

In [4]:
vote = VotingClassifier([
    ("lr",  make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))),
    ("rf",  RandomForestClassifier(random_state=42)),
    ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier())),
], voting="soft")

for name, m in [("Tree(free)", DecisionTreeClassifier(random_state=42)),
                ("RandomForest", RandomForestClassifier(random_state=42)),
                ("Voting", vote)]:
    acc = cross_val_score(m, X, y, cv=skf).mean()
    auc = cross_val_score(m, X, y, cv=skf, scoring="roc_auc").mean()
    print(f"{name:14s} acc={acc:.3f}  AUC={auc:.3f}")

Tree(free)     acc=0.910  AUC=0.900


RandomForest   acc=0.956  AUC=0.989


Voting         acc=0.972  AUC=0.995


### ข้อ 4 — importances

top-5: worst area 0.15 · worst concave points 0.13 · mean concave points 0.11 · worst radius 0.09 · worst perimeter 0.08
สอดคล้องการแพทย์ (ขนาด + ความหยักของขอบก้อนเนื้อ) แต่ "สำคัญต่อการแยกกลุ่มใน dataset นี้" ≠ "สาเหตุของโรค"
ฟีเจอร์ที่สัมพันธ์กันเอง (area/radius/perimeter!) ยังแบ่งกันกิน importance ด้วย

In [5]:
import pandas as pd
rf = RandomForestClassifier(random_state=42).fit(Xtr, ytr)
pd.Series(rf.feature_importances_, index=names).sort_values(ascending=False).head(5).round(3)

worst area              0.150
worst concave points    0.127
mean concave points     0.105
worst radius            0.087
worst perimeter         0.080
dtype: float64

### ข้อ 5 — hard vs soft

hard = นับเสียง label · **soft = เฉลี่ย probability แล้วตัดสิน** — soft ดีกว่าเพราะ "ความมั่นใจ" ของแต่ละโมเดลมีค่า
โมเดลที่ 90% มั่นใจควรมีน้ำหนักมากกว่าโมเดลที่เดา ๆ 51%

In [6]:
vote_hard = VotingClassifier([
    ("nb", make_pipeline(StandardScaler(), GaussianNB())),
    ("svm", make_pipeline(StandardScaler(), SVC(random_state=42))),
    ("rf", RandomForestClassifier(random_state=42))], voting="hard")
vote_soft = VotingClassifier([
    ("nb", make_pipeline(StandardScaler(), GaussianNB())),
    ("svm", make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))),
    ("rf", RandomForestClassifier(random_state=42))], voting="soft")

for name, m in [("hard", vote_hard), ("soft", vote_soft)]:
    acc = cross_val_score(m, X, y, cv=skf).mean()
    print(f"{name}: acc={acc:.3f}")

hard: acc=0.954


D:\Repositories\AI_course_MasterDegree\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\Repositories\AI_course_MasterDegree\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


D:\Repositories\AI_course_MasterDegree\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\Repositories\AI_course_MasterDegree\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


soft: acc=0.946


D:\Repositories\AI_course_MasterDegree\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
